### 1 b) Simulation of the development of Arterial Hypertension
Unlike aging (where the dominant factor is stiffness), **arterial hypertension** is mainly governed by a pathological increase in blood flow resistance due to vasoconstriction of small arteries.

To simulate this in the $WK_2$ model:
* We will strongly increase **Peripheral Resistance ($R_P$)** (e.g., 70% above the baseline value).
* We will slightly reduce **Compliance ($C_A$)** (e.g., by 20%) to account for the vascular damage associated with sustained high pressure.

The expected result is a global elevation of the pressure curve, affecting both systolic and diastolic pressure (increase in mean arterial pressure).

In [ ]:
# -------------------------------------------------------------------------
# GENERAL USE MODULES AND FLOW FUNCTION (EXCITATION)
# -------------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt

def Q_Sim(Qmax, T, t):
    """
    Generation of a FLOW signal (Half-sine) similar to human [ml/s]
    """
    Escala_ml = 1e5 * (Qmax / 8.8) # Adjustment to (500 ml/s)
    Q = Escala_ml * ((7.9853e-06 + 2.6617e-05*np.sin(2*np.pi*t/T+0.29498) +
                      2.3616e-05*np.sin(4*np.pi*t/T-1.1403) - 1.9016e-05*np.sin(6*np.pi*t/T+0.40435) -
                      8.5899e-06*np.sin(8*np.pi*t/T-1.1892) - 2.436e-06*np.sin(10*np.pi*t/T-1.4918) +
                      1.4905e-06*np.sin(12*np.pi*t/T+1.0536) + 1.3581e-06*np.sin(14*np.pi*t/T-0.47666) -
                      6.3031e-07*np.sin(16*np.pi*t/T+0.93768) - 4.5335e-07*np.sin(18*np.pi*t/T-0.79472) -
                      4.5184e-07*np.sin(20*np.pi*t/T-1.4095) - 5.6583e-07*np.sin(22*np.pi*t/T-1.3629) +
                      4.9522e-07*np.sin(24*np.pi*t/T+0.52495) + 1.3049e-07*np.sin(26*np.pi*t/T-0.97261) -
                      4.1072e-08*np.sin(28*np.pi*t/T-0.15685) - 2.4182e-07*np.sin(30*np.pi*t/T-1.4052) -
                      6.6217e-08*np.sin(32*np.pi*t/T-1.3785) - 1.5511e-07*np.sin(34*np.pi*t/T-1.2927) +
                      2.2149e-07*np.sin(36*np.pi*t/T+0.68178) + 6.7621e-08*np.sin(38*np.pi*t/T-0.98825) +
                      1.0973e-07*np.sin(40*np.pi*t/T+1.4327) - 2.5559e-08*np.sin(42*np.pi*t/T-1.2372) -
                      3.5079e-08*np.sin(44*np.pi*t/T+0.2328)))
    return Q

In [ ]:
# -------------------------------------------------------------------------
# NUMERICAL RESOLUTION OF THE WK2 MODEL (EULER'S METHOD)
# -------------------------------------------------------------------------

def resolver_wk2_euler(Q, t, Ca, Rp):
    """
    Solves the Differential Equation of the 2-element Windkessel
    dP/dt = Q/Ca - P/(Rp*Ca)
    """
    dt = t[1] - t[0]
    P = np.zeros(len(t))

    # Initial condition
    P[0] = 80.0

    for i in range(1, len(t)):
        dP_dt = (Q[i-1] / Ca) - (P[i-1] / (Rp * Ca))
        P[i] = P[i-1] + (dP_dt * dt)

    return P

In [ ]:
# -------------------------------------------------------------------------
# SIMULATION 1b: ARTERIAL HYPERTENSION
# -------------------------------------------------------------------------

# 1. Definition of the time and flow vector
Ts = 0.001
t = np.arange(0, 5, Ts)
T0 = 0.85
Qmax = 500

Q_entrada = Q_Sim(Qmax, T0, t)

# 2. Baseline Physiological Parameters
Ca_basal = 1.1  # ml/mmHg
Rp_basal = 0.9  # mmHg·s/ml

# 3. Parameters for Hypertension (Increase in Rp, slight drop in Ca)
Ca_hta = Ca_basal * 0.80
Rp_hta = Rp_basal * 1.70

# 4. Calculation of pressures
P_basal = resolver_wk2_euler(Q_entrada, t, Ca_basal, Rp_basal)
P_hta = resolver_wk2_euler(Q_entrada, t, Ca_hta, Rp_hta)

# 5. Visualization
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(t, Q_entrada, 'b', linewidth=2)
ax1.set_ylabel('Flow Q [ml/s]', fontsize=12)
ax1.set_title('Excitation: Left Ventricular Flow', fontsize=14)
ax1.grid(True, linestyle='--', alpha=0.7)

ax2.plot(t, P_basal, 'g', linewidth=2, label='Baseline Pressure (Healthy)')
ax2.plot(t, P_hta, 'orange', linewidth=2, label='Hypertension Pressure')
ax2.set_xlabel('Time [s]', fontsize=12)
ax2.set_ylabel('Aortic Pressure [mmHg]', fontsize=12)
ax2.set_title('Response: Effect of Hypertension on Aortic Pressure', fontsize=14)
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()

### Conclusion of the analysis (1 b)
Looking at the graphical results, we can draw the following conclusions from physiology and signal processing perspectives:

1. **DC Level Shift (Mean Pressure):** Unlike healthy aging where the curve changed its "shape" (greater ripple), in hypertension the most notable feature is that **the entire curve shifts upwards**. By significantly increasing Peripheral Resistance ($R_P$), the continuous component of the signal rises. Physiologically, this means that the diastolic pressure (the lowest point of the valley) is much higher than the normal 80 mmHg, forcing the left ventricle to overcome a much greater afterload in order to eject blood.
2. **RC Filter Effect:** From a circuit perspective, we increased the resistor of the low-pass filter. The steady-state voltage (Mean Pressure) is directly the product of the continuous current component (Mean Flow) and the resistance ($P_{med} = Q_{med} \cdot R_P$).
3. **Impact on Amplitude (Pulse Pressure):** Although the dominant effect is the global elevation, the concurrent reduction in compliance ($C_A$) due to vascular damage causes the difference between the systolic (peak) and diastolic (valley) pressures to also be wider than in the baseline case.